# Chapter 18 &mdash; The Problem of Recursion in an Anonymous World

**Concept 6 of the Chapter 18 decomposition:** *The Problem of Recursion in an Anonymous World*

`fact(n) = 1 if n==0 else n*fact(n-1)` names itself &mdash; which a lambda cannot do.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter18/Concept-Recursion-Without-Names/Concept-Recursion-Without-Names.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Recursion is defined by **self-reference**:

```
fact(n) = 1 if n == 0 else n * fact(n-1)
```

The right-hand side mentions `fact`. But a lambda is **anonymous** &mdash; it has no name
to mention. So how can the calculus express recursion at all?

The first move is to **abstract over the recursive call**: write a *non*-recursive
function $G$ that takes "the function to call recursively" as a parameter.

$$G = \lambda f.\lambda n.\ \text{if } n=0 \text{ then } 1 \text{ else } n\cdot f(n-1)$$

$G$ mentions no names of its own. What is needed now is an $f$ with $f = G\,f$ &mdash; a
**fixpoint** of $G$. That is what Concept 7 constructs.

## 2. Definitions

### The recursive version, and the abstracted one

In [ ]:
def fact_named(n):
    return 1 if n == 0 else n * fact_named(n - 1)

# G takes the recursive call as a parameter -- no self-reference
G_fact = lambda f: lambda n: 1 if n == 0 else n * f(n - 1)
G_fib  = lambda f: lambda n: n if n < 2 else f(n - 1) + f(n - 2)

### Unrolling G by hand, to see the fixpoint appear

In [ ]:
BOTTOM = lambda n: (_ for _ in ()).throw(RuntimeError("bottom: not defined here"))

def unroll(G, k):
    f = BOTTOM
    for _ in range(k):
        f = G(f)
    return f

## 3. Tests

The named version works, and names itself.

In [ ]:
print([fact_named(n) for n in range(7)])
body = "1 if n == 0 else n * fact_named(n - 1)"
print("\nits body is :", body)
print("does the body mention the function's own name? ", 'fact_named' in body)
assert 'fact_named' in body
print("That self-reference is exactly what a nameless lambda cannot write.")

$G$ mentions **nothing**. It is not recursive at all.

In [ ]:
print("G_fact = lambda f: lambda n: 1 if n == 0 else n * f(n-1)")
print()
print("G_fact is a perfectly ordinary two-argument function.")
print("  G_fact(anything)(0) =", G_fact(lambda k: 'unused')(0))
assert G_fact(lambda k: 'unused')(0) == 1

**Unrolling $G$ $k$ times** gives factorial correct up to $k$.

In [ ]:
for k in [1, 3, 5]:
    f = unroll(G_fact, k)
    ok = []
    for n in range(6):
        try: ok.append(f(n))
        except RuntimeError: ok.append('?')
    print("  G unrolled %d times : %s" % (k, ok))
f5 = unroll(G_fact, 5)
assert [f5(n) for n in range(5)] == [1, 1, 2, 6, 24]

More unrollings, more of the function. The limit is what we want.

In [ ]:
for k in [2, 4, 8]:
    f = unroll(G_fact, k)
    correct = 0
    for n in range(k):
        try:
            if f(n) == fact_named(n): correct += 1
        except RuntimeError: pass
    print("  %d unrollings -> correct on inputs 0..%d" % (k, correct - 1))
print("\nWe want the LIMIT: an f with f = G(f), correct on every input.")

A fixpoint is exactly that.

In [ ]:
f = unroll(G_fact, 10)
print("is G(f) the same as f, on 0..8 ?",
      all(G_fact(f)(n) == f(n) for n in range(8)))
assert all(G_fact(f)(n) == f(n) for n in range(8))
print()
print("f = G(f)  is the equation.  Concept 7 builds a combinator that")
print("SOLVES it, for any G, with no names anywhere.")

The same abstraction works for any recursive definition.

In [ ]:
f_fib = unroll(G_fib, 12)
print("fib via unrolled G :", [f_fib(n) for n in range(10)])
assert [f_fib(n) for n in range(10)] == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

## 4. Exercises


1. Write $G$ for the Ackermann function. How many parameters does it take?
2. How many unrollings does `G_fib` need to be correct on input $n$?
3. What does `BOTTOM` correspond to in denotational semantics?

In [ ]:
# Your work for the exercises above.